# Chopp & Cia · 03 — Análise Exploratória

**Projeto Integrador VI** · FATEC Votorantim · 2º Semestre/2026

Este notebook responde, em linguagem de negócio, a uma pergunta:

> **Quem são os clientes que dão prejuízo, e o que eles têm em comum?**

A empresa cede chopeiras em comodato e vende chopp a prazo. Isso cria dois riscos
que caminham juntos: o cliente **não paga** as parcelas, e o cliente **não devolve**
o equipamento. As análises abaixo medem os dois.

| | |
|:---|:---|
| **Entrada** | consolidado do notebook 01 (CSV) ou a tabela do 02 |
| **Saída** | leitura de negócio — este notebook **não altera dado nenhum** |
| **Ambiente** | Windows local ou Databricks |

---

### Como ler este notebook

Cada seção tem três partes, sempre na mesma ordem:

1. **A pergunta** — o que se quer saber, em português.
2. **O gráfico** — a resposta visual.
3. **A leitura** — o que o número significa para a operação, impresso abaixo do
   gráfico.

Você não precisa ler o código. As conclusões estão no texto e nos gráficos.

> **Sobre privacidade:** nenhum cliente é identificado por nome. A análise usa
> apenas o código do cliente (`ID_PESSOA`). Para saber de quem se trata,
> consulte o ERP com esse código.

## 1. De onde vêm os dados

Preencha o caminho conforme onde estiver rodando. No Databricks, informe a tabela
publicada pelo notebook 02; localmente, o CSV que o notebook 01 gerou.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

CAMINHO_CSV = r""          # local: CSV do notebook 01
TABELA = "projetointegrador.projetointegrador.dataset_consolidado_v1_0"   # Databricks

try:
    spark                                                      # noqa: F821
    EM_DATABRICKS = True
except NameError:
    EM_DATABRICKS = False

if EM_DATABRICKS:
    dados = spark.table(TABELA).toPandas()                     # noqa: F821
    origem = TABELA
else:
    from pathlib import Path
    if not CAMINHO_CSV.strip():
        raise ValueError("Preencha CAMINHO_CSV com o CSV gerado pelo notebook 01.")
    caminho = Path(CAMINHO_CSV.strip())
    if not caminho.exists():
        raise FileNotFoundError(f"CSV não encontrado: {caminho}")
    dados = pd.read_csv(caminho, sep=";", encoding="utf-8-sig", low_memory=False)
    origem = caminho.name

dados.columns = [str(c).strip().upper() for c in dados.columns]

NUMERICAS = [
    "DIAS_DESDE_PRIMEIRA_COMPRA", "DIAS_DESDE_ULTIMA_COMPRA", "FREQUENCIA_COMPRAS",
    "TOTAL_ITENS", "QTD_TOTAL_VENDIDA", "TOTAL_GASTO", "TICKET_MEDIO",
    "TOTAL_PARCELAS", "PARCELAS_ATRASADAS", "TAXA_ATRASO_PAGAMENTO",
    "MEDIA_DIAS_ATRASO_PAG", "MAX_DIAS_ATRASO_PAG", "VALOR_TOTAL_PARCELAS",
    "TOTAL_COMODATOS", "COMODATOS_ATRASADOS", "TAXA_ATRASO_COMODATO",
    "MEDIA_DIAS_ATRASO_COM", "MAX_DIAS_ATRASO_COM", "QTD_EQUIPAMENTOS",
    "RISCO_FINANCEIRO", "RISCO_COMODATO",
    "CORE_BUSINESS", "TEM_VENDAS", "TEM_FINANCEIRO", "TEM_COMODATO",
]
for coluna in NUMERICAS:
    dados[coluna] = pd.to_numeric(dados[coluna], errors="coerce")
for coluna in ["PRIMEIRA_COMPRA", "ULTIMA_COMPRA"]:
    dados[coluna] = pd.to_datetime(dados[coluna], errors="coerce")

print(f"Origem   : {origem}")
print(f"Carteira : {len(dados):,} clientes")

## 2. Como os gráficos são desenhados

As cores não são escolha de gosto. Elas seguem três regras que garantem que o
gráfico continue legível para quem tem daltonismo e quando impresso:

- **Severidade** (aging) usa **um tom só de azul**, do claro ao escuro — mais
  escuro é pior. A ordem das barras já carrega a informação; a cor reforça.
- **Categorias** (perfil de risco) usam cores separadas e testadas para não se
  confundirem sob daltonismo — e toda fatia leva rótulo escrito, nunca só cor.
- **Comparações de duas trilhas** (financeiro × comodato) usam azul e laranja,
  o par de maior separação visual.

In [ ]:
COR_TEXTO, COR_APOIO, COR_GRADE = "#0b0b0b", "#52514e", "#e3e2df"
COR_NEUTRA = "#b8b7b2"

# Rampa sequencial de um tom só: mais escuro = mais grave.
RAMPA_SEVERIDADE = ["#86b6ef", "#3987e5", "#256abf", "#184f95", "#0d366b"]

# Categóricas separadas sob protanopia e deuteranopia (verificado antes de usar).
CORES_PERFIL_RISCO = {
    "SEM RISCO": "#008300",
    "SÓ COMODATO": "#eda100",
    "SÓ FINANCEIRO": "#e87ba4",
    "RISCO DUPLO": "#4a3aa7",
}
COR_FINANCEIRO, COR_COMODATO = "#2a78d6", "#eb6834"

ORDEM_AGING = ["Sem Atraso", "1-3 Dias", "4-7 Dias", "8-15 Dias",
               "16-20 Dias", "21-30 Dias", "+30 Dias"]
ORDEM_PERFIL_RISCO = ["SEM RISCO", "SÓ COMODATO", "SÓ FINANCEIRO", "RISCO DUPLO"]

plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": COR_GRADE, "axes.labelcolor": COR_APOIO,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlecolor": COR_TEXTO,
    "axes.grid": True, "grid.color": COR_GRADE, "grid.linewidth": 0.8,
    "axes.axisbelow": True,          # grade ATRAS das barras
    "xtick.color": COR_APOIO, "ytick.color": COR_APOIO,
    "font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
})


def cor_severidade(i, n):
    """Escolhe o degrau da rampa proporcional à posição na escala de severidade."""
    if n <= 1:
        return RAMPA_SEVERIDADE[-1]
    passo = i * (len(RAMPA_SEVERIDADE) - 1) / (n - 1)
    return RAMPA_SEVERIDADE[int(round(passo))]


def moeda(valor, _=None):
    if valor >= 1e6:
        return f"R$ {valor / 1e6:.1f}M"
    if valor >= 1e3:
        return f"R$ {valor / 1e3:.0f}k"
    return f"R$ {valor:.0f}"


def milhar(valor, _=None):
    return f"{int(valor):,}".replace(",", ".")


def leitura(titulo, linhas):
    """Imprime a interpretação de negócio abaixo do gráfico."""
    print(f"\n{titulo}")
    print("-" * max(len(titulo), 40))
    for linha in linhas:
        print(f"  {linha}")


def rotular_barras(ax, barras, valores, formato=milhar, folga=0.01):
    """Escreve o valor ao lado de cada barra — identidade nunca fica só na cor."""
    limite = ax.get_xlim()[1]
    for barra, valor in zip(barras, valores):
        ax.text(barra.get_width() + limite * folga,
                barra.get_y() + barra.get_height() / 2,
                formato(valor), va="center", fontsize=9, color=COR_TEXTO)


print("Padrão visual carregado.")

## 3. O tamanho da carteira

Antes de falar de risco, é preciso saber de quantos clientes se está falando — e
quantos deles a empresa realmente conhece.

Nem todo cliente cadastrado comprou; nem todo comprador pegou equipamento em
comodato. Um cliente sem histórico não é um cliente de baixo risco: é um cliente
**sobre o qual não se sabe nada**, e a diferença importa na hora de conceder
crédito.

In [ ]:
total = len(dados)
cobertura = [
    ("Clientes cadastrados", total),
    ("Com alguma compra", int(dados["TEM_VENDAS"].sum())),
    ("Com parcelas a receber", int(dados["TEM_FINANCEIRO"].sum())),
    ("Com equipamento cedido", int(dados["TEM_COMODATO"].sum())),
    ("Compraram chopp/chopeira", int(dados["CORE_BUSINESS"].sum())),
]

fig, ax = plt.subplots(figsize=(9, 3.6))
rotulos = [r for r, _ in cobertura][::-1]
valores = [v for _, v in cobertura][::-1]
cores = [COR_NEUTRA] + [cor_severidade(i, 4) for i in range(4)][::-1]

barras = ax.barh(rotulos, valores, color=cores, height=0.62)
ax.xaxis.set_major_formatter(FuncFormatter(milhar))
ax.set_xlim(0, total * 1.18)
ax.set_title("Quantos clientes a empresa realmente conhece")
ax.grid(axis="y", visible=False)
rotular_barras(ax, barras, valores)
plt.tight_layout()
plt.show()

sem_historico = int(((dados["TEM_VENDAS"] == 0) | (dados["TOTAL_PARCELAS"] == 0)).sum())
leitura("O que isso significa", [
    f"A carteira tem {total:,} clientes cadastrados.",
    f"{int(dados['CORE_BUSINESS'].sum()):,} compraram chopp ou chopeira "
    f"({dados['CORE_BUSINESS'].mean() * 100:.0f}% do total) — é o público do negócio.",
    f"{sem_historico:,} clientes não têm compra ou não têm parcela registrada.",
    "Sobre esses últimos não há histórico para avaliar: não são bons pagadores,",
    "são desconhecidos. Conceder crédito a eles é uma decisão sem base em dado.",
])

## 4. Quanto da carteira está em risco

Um cliente entra em risco quando **mais de 20% dos seus compromissos atrasam** —
seja no pagamento das parcelas, seja na devolução do equipamento.

Os dois riscos são diferentes e exigem ações diferentes:

- **Risco financeiro** — o dinheiro não entra. Ação: cobrança, limite de crédito.
- **Risco de comodato** — a chopeira não volta. Ação: recolhimento, bloqueio de
  novo empréstimo.
- **Risco duplo** — as duas coisas ao mesmo tempo. É o grupo mais crítico: além
  de dever, o cliente está com um bem da empresa.

In [ ]:
perfil = dados["PERFIL_RISCO"].value_counts().reindex(ORDEM_PERFIL_RISCO).fillna(0).astype(int)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2),
                               gridspec_kw={"width_ratios": [1.25, 1]})

barras = ax1.barh(list(perfil.index)[::-1], list(perfil.values)[::-1],
                  color=[CORES_PERFIL_RISCO[p] for p in list(perfil.index)[::-1]],
                  height=0.6)
ax1.xaxis.set_major_formatter(FuncFormatter(milhar))
ax1.set_xlim(0, perfil.max() * 1.2)
ax1.set_title("Clientes por perfil de risco")
ax1.grid(axis="y", visible=False)
rotular_barras(ax1, barras, list(perfil.values)[::-1])

# O dinheiro exposto importa mais que a contagem: 10 clientes grandes em risco
# pesam mais que 100 pequenos.
exposicao = dados.groupby("PERFIL_RISCO")["TOTAL_GASTO"].sum().reindex(
    ORDEM_PERFIL_RISCO).fillna(0)
barras2 = ax2.barh(list(exposicao.index)[::-1], list(exposicao.values)[::-1],
                   color=[CORES_PERFIL_RISCO[p] for p in list(exposicao.index)[::-1]],
                   height=0.6)
ax2.xaxis.set_major_formatter(FuncFormatter(moeda))
ax2.xaxis.set_major_locator(plt.MaxNLocator(4))   # 4 marcas: os rótulos R$ são largos
ax2.set_xlim(0, exposicao.max() * 1.30)
ax2.set_title("Faturamento associado a cada perfil")
ax2.set_yticks([])
ax2.grid(axis="y", visible=False)
rotular_barras(ax2, barras2, list(exposicao.values)[::-1], formato=moeda, folga=0.02)

plt.tight_layout()
plt.show()

em_risco = int(perfil.drop("SEM RISCO").sum())
duplo = int(perfil["RISCO DUPLO"])
valor_risco = float(exposicao.drop("SEM RISCO").sum())
leitura("O que isso significa", [
    f"{em_risco:,} clientes ({em_risco / total * 100:.0f}% da carteira) atrasam mais de 20%",
    "dos seus compromissos.",
    f"Esses clientes respondem por {moeda(valor_risco)} de faturamento "
    f"({valor_risco / dados['TOTAL_GASTO'].sum() * 100:.0f}% do total).",
    "",
    f"{duplo:,} estão em RISCO DUPLO: devem dinheiro E estão com equipamento da",
    "empresa. É por onde a cobrança deve começar — é o grupo em que a empresa",
    "tem, ao mesmo tempo, o maior prejuízo e o maior poder de negociação.",
])

## 5. Há quanto tempo os atrasos se arrastam

Atraso de 2 dias e atraso de 6 meses são problemas diferentes. O primeiro é
esquecimento; o segundo, na prática, é perda.

O gráfico abaixo classifica cada cliente pela **pior** ocorrência do seu
histórico — não pela média. Um cliente que atrasou 45 dias uma vez é um caso de
"+30 Dias", ainda que a média o colocasse numa faixa branda: a média esconde
exatamente o evento que interessa.

In [ ]:
fig, eixos = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)

for ax, coluna, titulo, universo in [
    (eixos[0], "AGING_PAGAMENTO", "Atraso de pagamento", dados["TEM_FINANCEIRO"] == 1),
    (eixos[1], "AGING_COMODATO", "Atraso na devolução", dados["TEM_COMODATO"] == 1),
]:
    # Só quem tem a trilha entra na conta: cliente sem comodato não é "sem
    # atraso de comodato", ele simplesmente não tem comodato.
    faixas = (dados[universo][coluna].value_counts()
              .reindex(ORDEM_AGING).fillna(0).astype(int))
    cores = [cor_severidade(i, len(ORDEM_AGING)) for i in range(len(ORDEM_AGING))]
    barras = ax.bar(range(len(faixas)), faixas.values, color=cores, width=0.68)
    ax.set_xticks(range(len(faixas)))
    ax.set_xticklabels(faixas.index, rotation=45, ha="right", fontsize=9)
    ax.set_title(f"{titulo}  ({int(universo.sum()):,} clientes)")
    ax.yaxis.set_major_formatter(FuncFormatter(milhar))
    ax.grid(axis="x", visible=False)
    for barra, valor in zip(barras, faixas.values):
        if valor:
            ax.text(barra.get_x() + barra.get_width() / 2, valor + faixas.max() * 0.02,
                    milhar(valor), ha="center", fontsize=8.5, color=COR_TEXTO)
    ax.set_ylim(0, faixas.max() * 1.15)

eixos[0].set_ylabel("clientes")
plt.tight_layout()
plt.show()

com_fin = dados[dados["TEM_FINANCEIRO"] == 1]
com_com = dados[dados["TEM_COMODATO"] == 1]
graves_fin = int((com_fin["AGING_PAGAMENTO"] == "+30 Dias").sum())
graves_com = int((com_com["AGING_COMODATO"] == "+30 Dias").sum())
leitura("O que isso significa", [
    f"Pagamento: {graves_fin:,} clientes já atrasaram mais de 30 dias "
    f"({graves_fin / max(len(com_fin), 1) * 100:.0f}% dos que têm parcelas).",
    f"Comodato: {graves_com:,} clientes ficaram mais de 30 dias com o equipamento",
    f"além do prazo ({graves_com / max(len(com_com), 1) * 100:.0f}% dos que têm comodato).",
    "",
    "As barras à direita são as que exigem ação. Quanto mais à direita, menor a",
    "chance de recuperação espontânea — e maior o custo de continuar esperando.",
])

## 6. O risco tem endereço, forma de pagamento e tipo de negócio?

Se clientes de uma certa cidade, de um certo tipo de estabelecimento ou de uma
certa forma de pagamento atrasam sistematicamente mais, isso é acionável: dá para
ajustar política de crédito por grupo, sem esperar o cliente atrasar.

A linha tracejada é a média da carteira. Barra acima da linha = grupo pior que a
média. Só entram grupos com pelo menos 30 clientes — abaixo disso, a diferença
pode ser sorte.

In [ ]:
MINIMO_GRUPO = 30
media_geral = dados["RISCO_FINANCEIRO"].mean() * 100

fig, eixos = plt.subplots(1, 3, figsize=(13.5, 4.2))

for ax, dimensao, titulo in [
    (eixos[0], "CIDADE", "Por cidade"),
    (eixos[1], "PAGAMENTO", "Por forma de pagamento"),
    (eixos[2], "SEGMENTO", "Por tipo de estabelecimento"),
]:
    grupo = dados.groupby(dimensao).agg(
        clientes=("ID_PESSOA", "count"),
        risco=("RISCO_FINANCEIRO", "mean"),
    )
    grupo = grupo[grupo["clientes"] >= MINIMO_GRUPO].copy()
    grupo["risco"] *= 100
    grupo = grupo.sort_values("risco").tail(8)

    if grupo.empty:
        ax.text(0.5, 0.5, f"nenhum grupo com {MINIMO_GRUPO}+ clientes",
                ha="center", va="center", color=COR_APOIO, transform=ax.transAxes)
        ax.set_title(titulo)
        ax.set_axis_off()
        continue

    # Emphasis: o grupo pior que a média ganha cor; o resto é contexto.
    cores = [RAMPA_SEVERIDADE[-2] if v > media_geral else COR_NEUTRA
             for v in grupo["risco"]]
    rotulos = [f"{i[:22]}  (n={int(n)})" for i, n in zip(grupo.index, grupo["clientes"])]
    barras = ax.barh(rotulos, grupo["risco"].values, color=cores, height=0.62)
    ax.axvline(media_geral, color=COR_TEXTO, linestyle="--", linewidth=1.2)
    ax.set_xlim(0, max(grupo["risco"].max() * 1.25, media_geral * 1.3))
    ax.set_title(titulo)
    ax.set_xlabel("% de clientes em risco financeiro")
    ax.grid(axis="y", visible=False)
    ax.tick_params(axis="y", labelsize=8.5)
    for barra, valor in zip(barras, grupo["risco"].values):
        ax.text(valor + ax.get_xlim()[1] * 0.035, barra.get_y() + barra.get_height() / 2,
                f"{valor:.0f}%", va="center", fontsize=8.5, color=COR_TEXTO)

fig.suptitle(f"Risco financeiro por grupo — média da carteira: {media_geral:.0f}% "
             "(linha tracejada)", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

leitura("O que isso significa", [
    f"A média da carteira é {media_geral:.0f}% de clientes em risco financeiro.",
    "Os grupos coloridos ficam acima dessa média — são candidatos a uma política",
    "de crédito mais rígida (entrada maior, limite menor, prazo mais curto).",
    "",
    "Cuidado ao ler: grupo pequeno oscila muito. Por isso só aparecem aqui os",
    f"grupos com {MINIMO_GRUPO} clientes ou mais, e o n de cada um está no rótulo.",
    "Nenhuma dessas diferenças foi testada estatisticamente — são indícios para",
    "investigar, não conclusões fechadas.",
])

## 7. Cliente que sumiu é cliente que não pagou?

Uma suspeita comum na operação: o cliente que parou de comprar parou porque está
devendo. Se for verdade, o tempo desde a última compra vira um sinal de alerta
barato — não exige cálculo nenhum, está no extrato.

O gráfico testa a suspeita. Ele cruza há quanto tempo o cliente não compra com o
percentual dele que está em risco — e o resultado pode contrariar a intuição.

In [ ]:
com_compra = dados[dados["TEM_VENDAS"] == 1].copy()

FAIXAS = [(0, 30, "até 1 mês"), (30, 90, "1 a 3 meses"), (90, 180, "3 a 6 meses"),
          (180, 365, "6 a 12 meses"), (365, np.inf, "mais de 1 ano")]
com_compra["RECENCIA"] = pd.cut(
    com_compra["DIAS_DESDE_ULTIMA_COMPRA"],
    bins=[f[0] for f in FAIXAS] + [np.inf],
    labels=[f[2] for f in FAIXAS], right=False,
)

recencia = com_compra.groupby("RECENCIA", observed=False).agg(
    clientes=("ID_PESSOA", "count"),
    risco_financeiro=("RISCO_FINANCEIRO", "mean"),
    risco_comodato=("RISCO_COMODATO", "mean"),
)
recencia[["risco_financeiro", "risco_comodato"]] *= 100

fig, ax = plt.subplots(figsize=(10, 4.2))
x = np.arange(len(recencia))
largura = 0.38
ax.bar(x - largura / 2, recencia["risco_financeiro"], largura,
       label="Risco financeiro (não paga)", color=COR_FINANCEIRO)
ax.bar(x + largura / 2, recencia["risco_comodato"], largura,
       label="Risco de comodato (não devolve)", color=COR_COMODATO)

for i, (fin, com, n) in enumerate(zip(recencia["risco_financeiro"],
                                      recencia["risco_comodato"],
                                      recencia["clientes"])):
    ax.text(i - largura / 2, fin + 1.5, f"{fin:.0f}%", ha="center", fontsize=8.5,
            color=COR_TEXTO)
    ax.text(i + largura / 2, com + 1.5, f"{com:.0f}%", ha="center", fontsize=8.5,
            color=COR_TEXTO)

ax.set_xticks(x)
ax.set_xticklabels([f"{r}\n(n={int(n)})"
                    for r, n in zip(recencia.index, recencia["clientes"])])
ax.set_xlabel("tempo desde a última compra")
ax.set_ylabel("% de clientes em risco")
ax.set_title("O cliente que sumiu está devendo?")
ax.set_ylim(0, max(recencia[["risco_financeiro", "risco_comodato"]].max()) * 1.25)
ax.legend(frameon=False, loc="upper left")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

recentes = recencia["risco_financeiro"].iloc[0]
antigos = recencia["risco_financeiro"].iloc[-1]
sobe = antigos > recentes
leitura("O que isso significa", [
    f"Quem comprou {recencia.index[0]}: {recentes:.0f}% em risco financeiro.",
    f"Quem não compra há {recencia.index[-1]}: {antigos:.0f}%.",
    "",
    ("O risco CRESCE conforme o cliente some — sumiço e inadimplência andam"
     if sobe else
     "O risco CAI conforme o cliente some — o oposto da suspeita comum:"),
    ("juntos, e o tempo sem comprar serve de alerta precoce." if sobe else
     "quem ainda compra é justamente quem ainda tem conta aberta para atrasar."),
    "",
    ("" if sobe else
     "Faz sentido operacional: cliente ativo compra a prazo toda semana e tem"),
    ("" if sobe else
     "muitas parcelas correndo; quem parou já liquidou ou foi cortado. Ou seja,"),
    ("" if sobe else
     "tempo sem comprar NÃO serve de alerta de inadimplência nesta carteira."),
    "",
    "Em nenhum dos casos o dado mostra causa: não se sabe se o cliente sumiu",
    "porque devia, ou se deve porque sumiu. Serve para priorizar visita — não",
    "para afirmar o motivo.",
])

## 8. Onde está o dinheiro em jogo

As análises anteriores contam clientes. Esta conta **reais**.

Os dois lados importam: um cliente pequeno que nunca paga é um aborrecimento; um
cliente grande que atrasa é um problema de caixa. O gráfico separa os clientes em
quatro grupos de tamanho igual (25% cada) pelo faturamento, e mostra o risco em
cada um.

In [ ]:
com_faturamento = dados[dados["TOTAL_GASTO"] > 0].copy()
com_faturamento["PORTE"] = pd.qcut(
    com_faturamento["TOTAL_GASTO"], 4,
    labels=["25% menores", "25% médio-baixo", "25% médio-alto", "25% maiores"],
)

porte = com_faturamento.groupby("PORTE", observed=False).agg(
    clientes=("ID_PESSOA", "count"),
    faturamento=("TOTAL_GASTO", "sum"),
    risco=("RISCO_FINANCEIRO", "mean"),
)
porte["risco"] *= 100

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))

cores = [cor_severidade(i, len(porte)) for i in range(len(porte))]
barras = ax1.bar(range(len(porte)), porte["faturamento"], color=cores, width=0.66)
ax1.set_xticks(range(len(porte)))
ax1.set_xticklabels(porte.index, rotation=20, ha="right", fontsize=9)
ax1.yaxis.set_major_formatter(FuncFormatter(moeda))
ax1.set_title("Faturamento concentrado por porte de cliente")
ax1.grid(axis="x", visible=False)
for barra, valor in zip(barras, porte["faturamento"]):
    ax1.text(barra.get_x() + barra.get_width() / 2, valor + porte["faturamento"].max() * 0.02,
             moeda(valor), ha="center", fontsize=8.5, color=COR_TEXTO)
ax1.set_ylim(0, porte["faturamento"].max() * 1.15)

barras2 = ax2.bar(range(len(porte)), porte["risco"], color=cores, width=0.66)
ax2.set_xticks(range(len(porte)))
ax2.set_xticklabels(porte.index, rotation=20, ha="right", fontsize=9)
ax2.set_title("% em risco financeiro dentro de cada grupo")
ax2.set_ylabel("% de clientes em risco")
ax2.grid(axis="x", visible=False)
for barra, valor in zip(barras2, porte["risco"]):
    ax2.text(barra.get_x() + barra.get_width() / 2, valor + porte["risco"].max() * 0.03,
             f"{valor:.0f}%", ha="center", fontsize=8.5, color=COR_TEXTO)
ax2.set_ylim(0, porte["risco"].max() * 1.2)

plt.tight_layout()
plt.show()

topo = porte.iloc[-1]
fatia_topo = topo["faturamento"] / porte["faturamento"].sum() * 100
leitura("O que isso significa", [
    f"Os 25% maiores clientes concentram {fatia_topo:.0f}% do faturamento.",
    f"Dentro desse grupo, {topo['risco']:.0f}% estão em risco financeiro.",
    "",
    "É onde a cobrança rende mais por telefonema: são poucos clientes e muito",
    "dinheiro. Perder um cliente grande custa mais do que perder dezenas de",
    "pequenos — e, no grupo de cima, a régua de crédito não deveria ser a mesma.",
])

## 9. Quantas compras bastam para confiar no número?

Esta seção é a mais técnica, e ela existe por um motivo prático.

Se um cliente comprou **uma vez** e atrasou essa única parcela, a taxa de atraso
dele é 100%. O número está certo e não significa nada — uma amostra de tamanho 1
não descreve comportamento.

A tabela abaixo mostra o efeito de exigir um histórico mínimo: quantos clientes
sobram, e como a taxa de risco muda. É a decisão que o notebook 04 vai tomar ao
definir quem entra no modelo.

In [ ]:
print(f"{'mínimo de compras':>18} {'clientes':>10} {'% da carteira':>14} {'% em risco':>12}")
print("-" * 58)
for minimo in [0, 1, 2, 3, 5, 10]:
    elegiveis = dados[dados["FREQUENCIA_COMPRAS"] > minimo]
    if elegiveis.empty:
        continue
    print(f"{'> ' + str(minimo):>18} {len(elegiveis):>10,} "
          f"{len(elegiveis) / total * 100:>13.0f}% "
          f"{elegiveis['RISCO_FINANCEIRO'].mean() * 100:>11.0f}%")

uma_compra = dados[dados["FREQUENCIA_COMPRAS"] <= 1]
leitura("O que isso significa", [
    f"{len(uma_compra):,} clientes têm no máximo uma compra registrada.",
    "Para eles, qualquer taxa de atraso é 0% ou 100% — sem meio-termo possível.",
    "",
    "Exigir um histórico mínimo torna o número confiável, mas reduz a carteira",
    "analisável. É uma troca, e a escolha é de negócio, não de estatística:",
    "quanto risco de errar se aceita para poder atender mais clientes.",
])

## 10. Resumo

O que a exploração estabeleceu, e o que ela deliberadamente **não** estabelece.

In [ ]:
em_risco = int((dados["PERFIL_RISCO"] != "SEM RISCO").sum())
duplo = int((dados["PERFIL_RISCO"] == "RISCO DUPLO").sum())
exposto = float(dados[dados["PERFIL_RISCO"] != "SEM RISCO"]["TOTAL_GASTO"].sum())

print("=" * 66)
print(f"{'RESUMO DA CARTEIRA':^66}")
print("=" * 66)
print(f"  Clientes cadastrados          {total:>10,}")
print(f"  Compraram chopp/chopeira      {int(dados['CORE_BUSINESS'].sum()):>10,}")
print(f"  Em risco (algum tipo)         {em_risco:>10,}  ({em_risco / total * 100:.0f}%)")
print(f"  Em risco duplo                {duplo:>10,}  ({duplo / total * 100:.0f}%)")
print(f"  Faturamento exposto           {moeda(exposto):>10}")
print("=" * 66)

leitura("O que a EDA NÃO responde", [
    "1. Não há corte no tempo: as características e o atraso vêm do mesmo",
    "   período. Isso descreve o que já aconteceu — não prevê o próximo mês.",
    "",
    "2. Os grupos com mais risco são indícios, não causas. Nenhuma diferença",
    "   foi testada estatisticamente.",
    "",
    "3. 'Risco' aqui é a régua de 20% de atraso, uma escolha de negócio.",
    "   Mudar o percentual muda quem é considerado arriscado.",
    "",
    "O notebook 04 transforma essa leitura em alvo de modelo, e os 05 a 07",
    "verificam se o padrão é forte o bastante para ser aprendido.",
])